# 03 · Train Model B — the surrogate panel

Deliberately *not* hardened. Trained on the non-adversarial split only, so it
behaves like a typical third-party detector — which is the whole point.

The humanizer optimises against this model plus the perplexity ratio and the
stylometric features. Optimising against a panel rather than a single
classifier is what makes the rewrite generalise: tuned against one model it
learns that model's quirks; tuned against a diverse panel it learns the
properties third-party detectors genuinely share (PRD 8.1).

Same architecture, same loss, different data. Runtime: 4–8 hours.


In [ ]:
# Kaggle setup. Run once per session.
!pip install -q "transformers>=4.44" "datasets>=2.20" sentencepiece onnx onnxruntime \
    "optimum[onnxruntime]" pyarrow

import sys, os
from pathlib import Path

# The repo is added as a Kaggle dataset, or cloned. Point REPO at it.
REPO = Path("/kaggle/input/ai-detector-repo") if Path("/kaggle/input/ai-detector-repo").exists() \
       else Path("/kaggle/working/ai-detector")
if not REPO.exists():
    !git clone --depth 1 $GIT_URL /kaggle/working/ai-detector

sys.path.insert(0, str(REPO / "training"))
sys.path.insert(0, str(REPO / "api"))

WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)
DATA = WORK / "data"; DATA.mkdir(exist_ok=True)
MODELS = WORK / "models"; MODELS.mkdir(exist_ok=True)
print("repo:", REPO)


In [ ]:
import pandas as pd
from pathlib import Path
from lib.data import FEATURE_NAMES
from lib.train import TrainConfig, train

frame = pd.read_parquet(DATA / "train_b.parquet")

# Hold out by *generator and domain* where possible, not at random. A random
# split lets the model memorise a generator's quirks and score well on rows
# from the same generator, which is precisely the overfitting RAID exposed
# (E3: fine-tuned RoBERTa-Large averaged 56.7%).
holdout_domains = sorted(frame["domain"].unique())[-2:]
validation = frame[frame["domain"].isin(holdout_domains)]
training = frame[~frame["domain"].isin(holdout_domains)]
print(f"train {len(training):,} · validate {len(validation):,} on {holdout_domains}")

config = TrainConfig(
    backbone="microsoft/deberta-v3-base",
    max_length=768,
    batch_size=16,
    accumulation_steps=2,
    learning_rate=2e-5,
    epochs=3,
    fp16=True,
)

model, report = train(
    training, validation, list(FEATURE_NAMES),
    output_dir=WORK / "model_b",
    config=config,
)


In [ ]:
# Sanity check on what B is *for*. It should be clearly weaker than A on
# humanized text — that gap is the product's honesty margin, and the two
# numbers the UI shows are exactly this difference made visible.
final = report["final"]
print(f"AUROC {final['auroc']:.4f}  ·  TPR@1%FPR {final['tpr_at_1_fpr']:.4f}")
print("\nIf B matches A on adversarial text it is not a surrogate for anything.")
print("Notebook 05 measures that gap directly.")
